In [ ]:
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, 
                                     Dense, concatenate, multiply, add, Average)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# CONVOLUTIONAL BRANCH
def convolutional_branch(input_shape):
    input_layer = Input(shape=input_shape)

    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(input_layer)  # * Can adjust number of filters or kernel size
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)  # * Can experiment with different pooling strategies

    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)  # *
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)  # *

    conv3 = Conv2D(96, (3, 3), activation='relu', padding='same')(pool2)  # *
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)  # *

    conv4 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool3)  # *
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)  # *

    gap = GlobalAveragePooling2D()(pool4)  # * Can replace with flatten or other pooling
    dense_embedding = Dense(512, activation='relu')(gap)  # * Embedding size and activation can be tuned

    return Model(inputs=input_layer, outputs=dense_embedding)

# SIMPLE MUTUAL ATTENTION MECHANISM
def mutual_attention(features_main, features_inverse):
    weighted = multiply([features_main, features_inverse])  # * Can replace with more advanced attention mechanisms
    fused = add([weighted, features_inverse])
    return fused

# MAIN MODEL BUILDER 
def build_handwriting_verification_model(input_shape=(155, 220, 1)):
    input_text_ref = Input(shape=input_shape)
    input_text_query = Input(shape=input_shape)
    input_text_ref_inv = Input(shape=input_shape)
    input_text_query_inv = Input(shape=input_shape)

    shared_feature_extractor = convolutional_branch(input_shape)

    features_ref = shared_feature_extractor(input_text_ref)
    features_query = shared_feature_extractor(input_text_query)
    features_ref_inv = shared_feature_extractor(input_text_ref_inv)
    features_query_inv = shared_feature_extractor(input_text_query_inv)

    fused_ref = mutual_attention(features_ref, features_ref_inv)  # * Can adjust fusion logic
    fused_query = mutual_attention(features_query, features_query_inv)  # *

    pair_1 = concatenate([fused_ref, features_query])  # * Pairing strategies can be refined
    pair_2 = concatenate([features_ref, features_query])  # *
    pair_3 = concatenate([features_ref, fused_query])  # *
    pair_4 = concatenate([fused_ref, fused_query])  # *

    fc1 = Dense(128, activation='relu')(pair_1)  # * Can tune size, activation
    fc2 = Dense(128, activation='relu')(pair_2)  # *
    fc3 = Dense(128, activation='relu')(pair_3)  # *
    fc4 = Dense(128, activation='relu')(pair_4)  # *

    out1 = Dense(1, activation='sigmoid')(fc1)  # * Activation or binary decision structure can be tuned
    out2 = Dense(1, activation='sigmoid')(fc2)
    out3 = Dense(1, activation='sigmoid')(fc3)
    out4 = Dense(1, activation='sigmoid')(fc4)

    final_output = Average()([out1, out2, out3, out4])  # * Can replace with voting or trainable fusion

    model = Model(inputs=[
        input_text_ref, input_text_query,
        input_text_ref_inv, input_text_query_inv
    ], outputs=final_output)

    return model

# BUILD AND COMPILE
model = build_handwriting_verification_model()
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])  # * Tune optimiser or learning rate

model.summary()
